<a href="https://colab.research.google.com/github/canariocfet/Nexus/blob/main/Untitled4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import io
import re
import zipfile
import urllib.request
import pandas as pd
from IPython.display import display
from google.colab import data_table
from sklearn.feature_extraction.text import TfidfVectorizer
# Aprendizado supervisionado
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
# Aprendizado não supervisionado
from sklearn.cluster import KMeans

# Métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. DOWNLOAD DO DATASET FAKE.BR
# ============================================================

zip_url = (
    "https://codeload.github.com/"
    "roneysco/Fake.br-Corpus/zip/refs/heads/master" )

print("Baixando o dataset...")

req = urllib.request.Request(
    zip_url,
    headers={'User-Agent': 'Mozilla/5.0'} )

try:

    with urllib.request.urlopen(req) as response:

        zip_buffer = io.BytesIO(response.read())

        with zipfile.ZipFile(zip_buffer, 'r') as zip_ref:
            zip_ref.extractall('.')

    print("Download concluído com sucesso!\n")

except Exception as e:

    print(f"Erro ao baixar o arquivo: {e}")


# ============================================================
# CAMINHOS DOS ARQUIVOS
# ============================================================

fake_path = (
    'Fake.br-Corpus-master/'
    'full_texts/fake'
)

true_path = (
    'Fake.br-Corpus-master/'
    'full_texts/true'
)

fake_meta_path = (
    'Fake.br-Corpus-master/'
    'full_texts/fake-meta-information'
)

true_meta_path = (
    'Fake.br-Corpus-master/'
    'full_texts/true-meta-information'
)


dados = []


# ============================================================
# FUNÇÃO PARA LEITURA DOS METADADOS
# ============================================================

def ler_metadados(caminho_meta):

    """
    Lê as primeiras linhas do arquivo de meta-informação.
    """

    if os.path.exists(caminho_meta):

        with open(
            caminho_meta,
            'r',
            encoding='utf-8',
            errors='ignore'
        ) as f:

            linhas = [
                linha.strip()
                for linha in f.readlines()
            ]

            autor = (
                linhas[0]
                if len(linhas) > 0
                else "Não informado"
            )

            link = (
                linhas[1]
                if len(linhas) > 1
                else "Não informado"
            )

            categoria_tema = (
                linhas[2]
                if len(linhas) > 2
                else "Não informado"
            )

            data_pub = (
                linhas[3]
                if len(linhas) > 3
                else "Não informado"
            )

            return (
                autor,
                link,
                categoria_tema,
                data_pub
            )

    return (
        "N/A",
        "N/A",
        "N/A",
        "N/A"
    )


# ============================================================
# FUNÇÕES PARA EXTRAÇÃO DE CARACTERÍSTICAS
# ============================================================

def verificar_fontes(texto):

    texto_lower = texto.lower()

    termos_fonte = [
        "fonte",
        "segundo",
        "especialista",
        "pesquisa",
        "estudo",
        "afirmou",
        "de acordo",
        "g1",
        "r7",
        "uol"
    ]

    tem_aspas = (
        '"' in texto
        or '“' in texto
        or '”' in texto
    )

    tem_citacao = any(
        termo in texto_lower
        for termo in termos_fonte
    )

    return (
        "Presente"
        if (tem_aspas or tem_citacao)
        else "Ausente"
    )


def identificar_nome_fonte(texto):

    texto_lower = texto.lower()

    veiculos = [
        "g1",
        "r7",
        "uol",
        "folha",
        "estadao",
        "bbc",
        "reuters",
        "agencia brasil",
        "diario oficial",
        "aneel"
    ]

    for veiculo in veiculos:

        if veiculo in texto_lower:
            return veiculo.upper()

    termos = [
        "segundo",
        "especialista",
        "pesquisa",
        "estudo",
        "afirmou",
        "de acordo"
    ]

    if (
        any(
            termo in texto_lower
            for termo in termos
        )
        or '"' in texto
        or '“' in texto
    ):

        return "Citação/Aspas sem nome"

    return "Nenhuma fonte declarada"


def extrair_termos_sensacionalistas(texto):

    texto_lower = texto.lower()

    gatilhos = [
        "urgente",
        "bomba",
        "chocante",
        "surpreendente",
        "milagroso",
        "segredo",
        "inacreditável",
        "revelado",
        "compartilhe",
        "absurdo",
        "polêmica",
        "denúncia"
    ]

    encontrados = [
        palavra
        for palavra in gatilhos
        if palavra in texto_lower
    ]

    return (
        ", ".join(encontrados)
        if encontrados
        else "Nenhum termo encontrado"
    )


def extrair_frases_emotivas(texto):

    frases = re.findall(
        r'[^.!?]*[!?]',
        texto
    )

    if frases:

        trechos = [
            frase.strip()
            for frase in frases[:3]
        ]

        return " | ".join(trechos)

    return "Sem pontuação emotiva"


# ============================================================
# 2. CARREGAMENTO DAS NOTÍCIAS FALSAS
# ============================================================

for arquivo in os.listdir(fake_path)[:10]:

    caminho = os.path.join(
        fake_path,
        arquivo
    )

    if (
        os.path.isfile(caminho)
        and arquivo.endswith('.txt')
    ):

        nome_meta = arquivo.replace(
            '.txt',
            '-meta.txt'
        )

        caminho_meta = os.path.join(
            fake_meta_path,
            nome_meta
        )

        (
            autor,
            link,
            categoria_tema,
            data_pub
        ) = ler_metadados(caminho_meta)

        with open(
            caminho,
            'r',
            encoding='utf-8',
            errors='ignore'
        ) as f:

            conteudo = f.read()

            dados.append({

                'texto':
                    conteudo,

                'categoria':
                    'Fake',

                'label':
                    0,

                'autor':
                    autor,

                'link':
                    link,

                'categoria_tema':
                    categoria_tema,

                'data_publicacao':
                    data_pub,

                'fontes':
                    verificar_fontes(
                        conteudo
                    ),

                'fonte_identificada':
                    identificar_nome_fonte(
                        conteudo
                    ),

                'sensacionalismo':
                    conteudo.lower().count(
                        "urgente"
                    )
                    + conteudo.lower().count(
                        "bomba"
                    )
                    + conteudo.lower().count(
                        "chocante"
                    )
                    + conteudo.lower().count(
                        "segredo"
                    ),

                'quais_termo_sensacionalistas':
                    extrair_termos_sensacionalistas(
                        conteudo
                    ),

                'pontuacao_emotiva':
                    conteudo.count('!')
                    + conteudo.count('?'),

                'quais_frases_emotivas':
                    extrair_frases_emotivas(
                        conteudo
                    )

            })


# ============================================================
# CARREGAMENTO DAS NOTÍCIAS VERDADEIRAS
# ============================================================

for arquivo in os.listdir(true_path)[:10]:

    caminho = os.path.join(
        true_path,
        arquivo
    )

    if (
        os.path.isfile(caminho)
        and arquivo.endswith('.txt')
    ):

        nome_meta = arquivo.replace(
            '.txt',
            '-meta.txt'
        )

        caminho_meta = os.path.join(
            true_meta_path,
            nome_meta
        )

        (
            autor,
            link,
            categoria_tema,
            data_pub
        ) = ler_metadados(caminho_meta)

        with open(
            caminho,
            'r',
            encoding='utf-8',
            errors='ignore'
        ) as f:

            conteudo = f.read()

            dados.append({

                'texto':
                    conteudo,

                'categoria':
                    'True',

                'label':
                    1,

                'autor':
                    autor,

                'link':
                    link,

                'categoria_tema':
                    categoria_tema,

                'data_publicacao':
                    data_pub,

                'fontes':
                    verificar_fontes(
                        conteudo
                    ),

                'fonte_identificada':
                    identificar_nome_fonte(
                        conteudo
                    ),

                'sensacionalismo':
                    conteudo.lower().count(
                        "urgente"
                    )
                    + conteudo.lower().count(
                        "bomba"
                    )
                    + conteudo.lower().count(
                        "chocante"
                    )
                    + conteudo.lower().count(
                        "segredo"
                    ),

                'quais_termo_sensacionalistas':
                    extrair_termos_sensacionalistas(
                        conteudo
                    ),

                'pontuacao_emotiva':
                    conteudo.count('!')
                    + conteudo.count('?'),

                'quais_frases_emotivas':
                    extrair_frases_emotivas(
                        conteudo
                    )

            })


# Criando DataFrame
df_amostra = pd.DataFrame(dados)


print(
    f"Total de notícias carregadas: "
    f"{len(df_amostra)}"
)

print(
    df_amostra['categoria']
    .value_counts()
)


# ============================================================
# 3. TRANSFORMAÇÃO DOS TEXTOS COM TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=1000
)

X_tfidf = vectorizer.fit_transform(
    df_amostra['texto']
)


print(
    "\nFormato da matriz TF-IDF:",
    X_tfidf.shape
)


# ============================================================
# 4. APRENDIZADO SUPERVISIONADO
# LOGISTIC REGRESSION
# ============================================================

print(
    "\n========================================"
)

print(
    "APRENDIZADO SUPERVISIONADO"
)

print(
    "LOGISTIC REGRESSION"
)

print(
    "========================================\n"
)


# ============================================================
# Separação entre treinamento e teste
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X_tfidf,
        df_amostra['label'],
        test_size=0.30,
        random_state=42,
        stratify=df_amostra['label']
    )
)


print(
    "Quantidade de exemplos de treinamento:",
    X_train.shape[0]
)

print(
    "Quantidade de exemplos de teste:",
    X_test.shape[0]
)


# ============================================================
# Treinamento
# ============================================================

modelo = LogisticRegression(
    random_state=42
)

modelo.fit(
    X_train,
    y_train
)


# ============================================================
# Previsões
# ============================================================

y_pred = modelo.predict(
    X_test
)

y_proba = modelo.predict_proba(
    X_test
)[:, 1]


# ============================================================
# 5. MÉTRICAS DO MODELO SUPERVISIONADO
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_proba
)


print(
    "\n========================================"
)

print(
    "MÉTRICAS DO MODELO"
)

print(
    "========================================\n"
)


print(
    f"Accuracy : {accuracy:.2f}"
)

print(
    f"Precision: {precision:.2f}"
)

print(
    f"Recall   : {recall:.2f}"
)

print(
    f"F1-Score : {f1:.2f}"
)

print(
    f"ROC-AUC  : {roc_auc:.2f}"
)


# ============================================================
# MATRIZ DE CONFUSÃO
# ============================================================

matriz_confusao = confusion_matrix(
    y_test,
    y_pred
)


print(
    "\n========================================"
)

print(
    "MATRIZ DE CONFUSÃO"
)

print(
    "========================================\n"
)

print(
    matriz_confusao
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(
    "\n========================================"
)

print(
    "RELATÓRIO DE CLASSIFICAÇÃO"
)

print(
    "========================================\n"
)


print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            'Fake',
            'True'
        ],
        zero_division=0
    )
)


# ============================================================
# Probabilidades para todas as notícias
#
# Usado apenas para exibição na tabela final.
# ============================================================

probabilidades = modelo.predict_proba(
    X_tfidf
)


df_amostra[
    'porcentagem_verdade'
] = [

    f"{round(p[1] * 100, 1)}%"

    for p in probabilidades
]


df_amostra[
    'porcentagem_fake'
] = [

    f"{round(p[0] * 100, 1)}%"

    for p in probabilidades
]


# ============================================================
# 6. APRENDIZADO NÃO SUPERVISIONADO
# K-MEANS
# ============================================================

print(
    "\n========================================"
)

print(
    "APRENDIZADO NÃO SUPERVISIONADO"
)

print(
    "K-MEANS"
)

print(
    "========================================\n"
)


# O K-Means recebe SOMENTE os vetores TF-IDF.
#
# Ele não recebe:
# - categoria
# - label
#
# Portanto, tenta descobrir grupos automaticamente.

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)


clusters = kmeans.fit_predict(
    X_tfidf
)


# Adicionando o cluster ao DataFrame
df_amostra[
    'cluster_kmeans'
] = clusters


# ============================================================
# RESULTADOS DO K-MEANS
# ============================================================

print(
    "Quantidade de notícias por cluster:\n"
)


print(
    df_amostra[
        'cluster_kmeans'
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# Comparação dos clusters com a categoria real
#
# IMPORTANTE:
#
# Fake/True não foram usados no treinamento do K-Means.
#
# A comparação abaixo ocorre somente DEPOIS
# que os clusters foram criados.
# ============================================================

print(
    "\nComparação entre cluster "
    "e categoria real:\n"
)


comparacao_clusters = pd.crosstab(
    df_amostra['cluster_kmeans'],
    df_amostra['categoria']
)


display(
    comparacao_clusters
)


# ============================================================
# 7. ORGANIZAÇÃO DAS COLUNAS
# ============================================================

df_amostra = df_amostra[[

    'texto',

    'categoria',

    'cluster_kmeans',

    'autor',

    'link',

    'categoria_tema',

    'data_publicacao',

    'fontes',

    'fonte_identificada',

    'sensacionalismo',

    'quais_termo_sensacionalistas',

    'pontuacao_emotiva',

    'quais_frases_emotivas',

    'porcentagem_verdade',

    'porcentagem_fake'

]]


# ============================================================
# 8. EXIBIÇÃO DA TABELA INTERATIVA
# ============================================================

data_table.enable_dataframe_formatter()

# display( df_amostra )

Baixando o dataset...
Download concluído com sucesso!

Total de notícias carregadas: 20
categoria
Fake    10
True    10
Name: count, dtype: int64

Formato da matriz TF-IDF: (20, 1000)

APRENDIZADO SUPERVISIONADO
LOGISTIC REGRESSION

Quantidade de exemplos de treinamento: 14
Quantidade de exemplos de teste: 6

MÉTRICAS DO MODELO

Accuracy : 0.67
Precision: 0.60
Recall   : 1.00
F1-Score : 0.75
ROC-AUC  : 0.56

MATRIZ DE CONFUSÃO

[[1 2]
 [0 3]]

RELATÓRIO DE CLASSIFICAÇÃO

              precision    recall  f1-score   support

        Fake       1.00      0.33      0.50         3
        True       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6


APRENDIZADO NÃO SUPERVISIONADO
K-MEANS

Quantidade de notícias por cluster:

cluster_kmeans
0     2
1    18
Name: count, dtype: int64

Comparação entre cluster e categoria real:



categoria,Fake,True
cluster_kmeans,,
0,1,1
1,9,9
